# 第 7 章 · 多 agent（CrewAI · Expert Teams）

两个角色（研究员 + 撰稿人）串成内容生产流水线。

关键在 `Task(context=[research_task])`：**后一个 task 能把前一个的输出当上下文**——这是 CrewAI 版的交接。

kernel 选 `Python (agentic-crewai)`。

In [1]:
import os
import threading

from crewai import Agent, Task, Crew, Process, LLM

# 【适配】书里用 ChatGoogleGenerativeAI；CrewAI 1.15 不收 LangChain 对象，改用内置 deepseek provider
llm = LLM(model="deepseek/deepseek-flash",
          api_key=os.environ["OPENAI_API_KEY"],
          base_url="https://api.deepseek.com/v1")

researcher = Agent(
    role='Senior Research Analyst',
    goal='Find and summarize the latest trends in AI.',
    backstory="You are an experienced research analyst with a knack for identifying key trends.",
    verbose=False,
    allow_delegation=False,
    llm=llm,
)

writer = Agent(
    role='Technical Content Writer',
    goal='Write a clear and engaging blog post based on research findings.',
    backstory="You are a skilled writer who can translate complex technical topics into accessible content.",
    verbose=False,
    allow_delegation=False,
    llm=llm,
)

research_task = Task(
    description="Research the top 3 emerging trends in AI. Focus on practical applications.",
    expected_output="A detailed summary of the top 3 AI trends.",
    agent=researcher,
)

# ★ 本章新增 context：把上一个 task 的结果喂给这个 task —— CrewAI 的"交接"
writing_task = Task(
    description="Write a short blog post (about 200 words) based on the research findings.",
    expected_output="A complete blog post about the latest AI trends.",
    agent=writer,
    context=[research_task],
)

blog_creation_crew = Crew(
    agents=[researcher, writer],
    tasks=[research_task, writing_task],
    process=Process.sequential,
)


# 【适配】kernel 里有运行中的事件循环，同步 kickoff() 会抛 RuntimeError → 独立线程（同第 5 章）
def run_in_thread(fn):
    box = {}

    def _target():
        try:
            box["result"] = fn()
        except BaseException as e:               # noqa: BLE001
            box["error"] = e

    th = threading.Thread(target=_target)
    th.start()
    th.join()
    if "error" in box:
        raise box["error"]
    return box["result"]


print("## Running the blog creation crew ##")
result = run_in_thread(blog_creation_crew.kickoff)
print("\n## Final Output ##\n")
print(str(result)[:1200])

## Running the blog creation crew ##



## Final Output ##

# Beyond the Chatbot: Three AI Trends That Actually Ship

For the past two years, most AI conversation has centered on general-purpose chatbots. In 2026, the action has moved to production systems that **act, run cheaply, and live inside real business workflows**. Three trends stand out.

**1. Agentic AI.** Agents don't just answer questions — they plan multi-step tasks, call tools, and execute. They're resolving support tickets, triaging IT incidents, reconciling invoices, and coordinating supply chains. The pattern that works: narrow workflows, bounded permissions, human approval for high-risk actions, and hard metrics like cost per task and error rate. It's early production — with measurable ROI.

**2. Small, specialized, and edge models.** Not every task needs a frontier model. Compact, domain-tuned models deliver lower latency, lower cost, and stronger privacy — running on devices, factory lines, and in clinical settings. High-volume routine work goes small; o